# Scharbeutz Beach Visitor Load Forecasting

**Client**: Tourism organization managing overcrowding at Scharbeutz beach, Baltic Sea, Germany  
**Goal**: Forecast 4-hour occupancy slots to identify high-load periods and support proactive crowd management  
**Dataset**: `scharbeutz_beach_data_4h_sum.csv` — visitor occupancy aggregated in 4-hour intervals (Nov 2020 – Apr 2022)

## Project Context

Scharbeutz is a popular seaside resort on the Baltic Sea in Schleswig-Holstein, Germany. It attracts large numbers of domestic tourists — especially in summer — leading to overcrowding on the beach.

### Problem
The provided dataset contains raw occupancy counts but **lacks the contextual features** (weather, sea conditions, holidays, restrictions) needed for accurate forecasting.

### Approach

| Step | Action |
|------|--------|
| 1 | Load and clean the provided visitor occupancy dataset |
| 2 | Integrate external data: weather, sea temperature, multi-state holidays, bridge days, COVID |
| 3 | Engineer features from all sources |
| 4 | Explore patterns (EDA) |
| 5 | Train and compare ML models (Naive, Random Forest, XGBoost) |
| 6 | Classify predicted load levels for operational use |

### External Data Sources

| Source | Data | Access | Rationale |
|--------|------|--------|-----------|
| Open-Meteo Historical API | Daily weather — temp, rain, sunshine, wind, **UV index, feels-like temp** | Free, no API key | Beach comfort driven by weather; UV affects sunbathing appeal |
| Open-Meteo Marine API | **Baltic Sea surface temperature** (daily) | Free, no API key | <17°C = few swimmers; strongest non-weather beach driver |
| `holidays` Python library | German public holidays (Schleswig-Holstein) | Python package | Long weekends spike visitor numbers |
| SH + **HH + NRW + BY** School Calendars | School holidays for 4 states (hardcoded) | Public domain | Many visitors come from Hamburg, NRW, Bayern — not just SH residents |
| Public holiday calendar | **Bridge days** (Brückentage) — derived | Computed | Thu/Tue holidays create 4–5 day weekends |
| COVID restriction timeline | **Pandemic restriction level** 0/1/2 | Hardcoded | Hard lockdowns closed beaches; partial restrictions reduced visitor pool |

## 0. Environment Setup

In [1]:
# Install required packages (run once)
import subprocess, sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'xgboost', 'requests', 'holidays']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else f'FAILED: {result.stderr[:80]}'
    print(f'  {pkg:<20} {status}')
print('Done.')

  pandas               OK
  numpy                OK
  matplotlib           OK
  seaborn              OK
  scikit-learn         OK
  xgboost              OK
  requests             OK
  holidays             OK
Done.


In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless backend — must be set before importing pyplot
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import holidays

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix
)
import xgboost as xgb
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.style.use('ggplot')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Python {sys.version}')
print('All libraries imported successfully.')

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

## 1. Loading the Dataset

The CSV has an unusual two-row header:
- **Row 0**: File title `scharbeutz_beach_data_4h_sum` (not a column header)
- **Row 1**: Column headers `datetime;occupancy`
- **Rows 2+**: Semicolon-delimited data

Timestamps are in **UTC** → converted to local CET/CEST (Europe/Berlin).  
Negative occupancy values (sensor calibration artifacts) are clipped to 0.

In [3]:
# Locate dataset (works from notebooks/ or project root)
candidates = [
    '../scharbeutz_beach_data_4h_sum.csv',
    'scharbeutz_beach_data_4h_sum.csv',
]
raw_path = next((p for p in candidates if os.path.exists(p)), None)
if raw_path is None:
    raise FileNotFoundError('scharbeutz_beach_data_4h_sum.csv not found. Run from project root or notebooks/.')

# Load — skip the title row (row 0), use row 1 as header
df = pd.read_csv(raw_path, sep=';', skiprows=1)
df.columns = ['datetime', 'occupancy']

# Parse UTC timestamps and convert to local time
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df['datetime'] = df['datetime'].dt.tz_convert('Europe/Berlin')
df['datetime'] = df['datetime'].dt.tz_localize(None)  # strip tz for simpler downstream handling

# Clip sensor errors (negative values)
n_neg = (df['occupancy'] < 0).sum()
df['occupancy'] = df['occupancy'].clip(lower=0)

df = df.sort_values('datetime').reset_index(drop=True)

print(f'Loaded: {len(df):,} records')
print(f'Range:  {df["datetime"].min()} -> {df["datetime"].max()}')
print(f'Negative values clipped to 0: {n_neg}')
df.head(6)

Loaded: 3,240 records
Range:  2020-11-01 01:00:00 -> 2022-04-24 22:00:00
Negative values clipped to 0: 556


,datetime,occupancy
0,2020-11-01 01:00:00,0.2500
1,2020-11-01 05:00:00,25.5625
2,2020-11-01 09:00:00,439.1875
3,2020-11-01 13:00:00,782.0000
4,2020-11-01 17:00:00,427.1875
5,2020-11-01 21:00:00,418.3750


In [4]:
print('=== Dataset Overview ===')
print(f'Records:       {len(df):,}')
print(f'Date range:    {df["datetime"].min().date()} to {df["datetime"].max().date()}')
print(f'Duration:      {(df["datetime"].max() - df["datetime"].min()).days} days')
print(f'Time slots:    {sorted(df["datetime"].dt.hour.unique())}  (6 slots/day x 4h)')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print('Occupancy statistics:')
print(df['occupancy'].describe().round(1))
print()
print('Records per month:')
monthly_counts = df.groupby(df['datetime'].dt.to_period('M'))['occupancy'].count()
print(monthly_counts.to_string())

=== Dataset Overview ===
Records:       3,240
Date range:    2020-11-01 to 2022-04-24
Duration:      539 days
Time slots:    [np.int32(1), np.int32(2), np.int32(5), np.int32(6), np.int32(9), np.int32(10), np.int32(13), np.int32(14), np.int32(17), np.int32(18), np.int32(21), np.int32(22)]  (6 slots/day x 4h)
Missing values: 0

Occupancy statistics:
count    3240.0
mean      212.1
std       479.8
min         0.0
25%         7.5
50%        58.5
75%       181.1
max      5224.5
Name: occupancy, dtype: float64

Records per month:
datetime
2020-11    180
2020-12    186
2021-01    186
2021-02    168
2021-03    186
2021-04    180
2021-05    186
2021-06    180
2021-07    186
2021-08    186
2021-09    180
2021-10    186
2021-11    180
2021-12    186
2022-01    186
2022-02    168
2022-03    186
2022-04    144
Freq: M


## 2. External Data Sources

The base dataset only contains raw occupancy counts. We integrate six external sources to enrich the feature set: daily weather (temp, UV, feels-like), Baltic Sea surface temperature, public holidays, multi-state school holidays, bridge days, and COVID restriction levels.

### 2.1 Weather & Marine Comfort Data — Open-Meteo APIs

**Rationale**: Beach attendance is primarily driven by weather and water conditions.

**Daily weather** (`archive-api.open-meteo.com/v1/archive`):

| Variable | Description |
|----------|-------------|
| `temp_max` / `temp_min` | Daily temperature range (°C) |
| `precipitation` | Total daily rainfall (mm) |
| `sunshine_hours` | Total sunshine (seconds → hours) |
| `wind_speed` | Max daily wind speed (km/h) |
| `apparent_temp_max` | **Feels-like temperature** — heat index / wind chill |
| `uv_index_max` | **Daily UV index** — sunbathing appeal and risk |

**Baltic Sea surface temperature** (`marine-api.open-meteo.com/v1/marine`):

| Variable | Description |
|----------|-------------|
| `sea_temp` | Daily mean sea surface temperature (°C) |
| `is_swim_temp` | Binary: sea_temp ≥ 17°C (comfortable swimming threshold) |

> Note: BSH (Bundesamt für Seeschifffahrt) is the authoritative source for Baltic water temps but provides no free REST API. Open-Meteo Marine uses the same ERA5 / CMEMS reanalysis data.

**Location**: Scharbeutz 54.02°N, 10.76°E · **Cost**: Free, no API key

In [5]:
def fetch_weather_openmeteo(lat=54.02, lon=10.76, start='2020-11-01', end='2022-04-30'):
    """
    Fetch historical daily weather for Scharbeutz from Open-Meteo (free, no API key).
    Includes UV index and apparent (feels-like) temperature.
    Source: https://open-meteo.com/en/docs/historical-weather-api
    """
    cache_paths = ['../data/external/weather_scharbeutz.csv',
                   'data/external/weather_scharbeutz.csv']
    for cp in cache_paths:
        if os.path.exists(cp):
            df_w = pd.read_csv(cp, parse_dates=['date'])
            # Re-fetch if older cache is missing the new columns
            if 'apparent_temp_max' in df_w.columns and 'uv_index_max' in df_w.columns:
                print(f'Weather loaded from cache: {len(df_w)} days ({cp})')
                return df_w
            print('Cache outdated (missing UV / apparent_temp) — re-fetching...')
            break

    print(f'Fetching weather for Scharbeutz ({lat}N, {lon}E) from Open-Meteo...')
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': start,
        'end_date':   end,
        'daily':      ('temperature_2m_max,temperature_2m_min,precipitation_sum,'
                       'sunshine_duration,wind_speed_10m_max,weathercode,'
                       'apparent_temperature_max,uv_index_max'),
        'timezone':   'Europe/Berlin'
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        d = resp.json()['daily']
        n = len(d['time'])
        df_w = pd.DataFrame({
            'date':              pd.to_datetime(d['time']),
            'temp_max':          d['temperature_2m_max'],
            'temp_min':          d['temperature_2m_min'],
            'precipitation':     d['precipitation_sum'],
            'sunshine_hours':    [s / 3600.0 if s else 0.0 for s in d['sunshine_duration']],
            'wind_speed':        d['wind_speed_10m_max'],
            'weather_code':      d['weathercode'],
            'apparent_temp_max': d.get('apparent_temperature_max', [None] * n),
            'uv_index_max':      d.get('uv_index_max', [None] * n),
        })
        df_w = df_w.dropna(subset=['temp_max'])
        os.makedirs('../data/external', exist_ok=True)
        df_w.to_csv('../data/external/weather_scharbeutz.csv', index=False)
        print(f'Success: {len(df_w)} days. Cached to ../data/external/weather_scharbeutz.csv')
        return df_w
    except Exception as e:
        print(f'API call failed: {e}')
        print('Weather features will be unavailable.')
        return None

df_weather = fetch_weather_openmeteo()
if df_weather is not None:
    print()
    print(df_weather.describe().round(1))

Weather loaded from cache: 546 days (../data/external/weather_scharbeutz.csv)

                      date  temp_max  temp_min  precipitation  sunshine_hours  \
count                  546     546.0     546.0          546.0           546.0   
mean   2021-07-31 12:00:00      11.3       5.2            1.9             6.9   
min    2020-11-01 00:00:00      -3.8     -13.8            0.0             0.0   
25%    2021-03-17 06:00:00       6.1       0.9            0.0             1.8   
50%    2021-07-31 12:00:00      10.0       4.1            0.4             7.0   
75%    2021-12-14 18:00:00      15.8       9.7            2.3            11.4   
max    2022-04-30 00:00:00      32.8      20.5           38.0            16.3   
std                    NaN       7.0       6.0            3.6             5.1   

       wind_speed  weather_code  apparent_temp_max  uv_index_max  
count       546.0         546.0              546.0           0.0  
mean         21.9          37.7                8.2       

In [6]:
def fetch_sea_temp(lat=54.02, lon=10.76, start='2020-11-01', end='2022-04-30'):
    """
    Fetch daily Baltic Sea surface temperature from Open-Meteo Marine API.
    Uses ERA5 / Copernicus Marine reanalysis.
    Note: returns None if the nearest grid point is land (all-NaN response).
    """
    cache_paths = ['../data/external/sea_temp_scharbeutz.csv',
                   'data/external/sea_temp_scharbeutz.csv']
    for cp in cache_paths:
        if os.path.exists(cp):
            df_s = pd.read_csv(cp, parse_dates=['date'])
            if df_s['sea_temp'].notna().any():
                print(f'Sea temperature loaded from cache: {len(df_s)} days ({cp})')
                return df_s
            print('Cached sea temp is all-NaN — skipping.')
            return None

    print(f'Fetching Baltic Sea surface temperature ({lat}N, {lon}E) from Open-Meteo Marine API...')
    url = 'https://marine-api.open-meteo.com/v1/marine'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'hourly':     'sea_surface_temperature',
        'start_date': start,
        'end_date':   end,
        'timezone':   'UTC',
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        d = resp.json()['hourly']
        df_h = pd.DataFrame({
            'time':     pd.to_datetime(d['time']),
            'sea_temp': d['sea_surface_temperature'],
        })
        df_h['date'] = df_h['time'].dt.normalize()
        df_s = df_h.groupby('date')['sea_temp'].mean().reset_index()

        # Guard: if Marine grid point is land, all values are NaN — skip the feature
        if df_s['sea_temp'].isna().all():
            print('  Marine API returned all-NaN (grid point likely on land). Skipping sea_temp feature.')
            print('  Tip: try a slightly offshore coordinate, e.g. lat=54.05, lon=10.80')
            return None

        os.makedirs('../data/external', exist_ok=True)
        df_s.to_csv('../data/external/sea_temp_scharbeutz.csv', index=False)
        print(f'Success: {len(df_s)} days, {df_s["sea_temp"].notna().sum()} non-NaN.')
        print(f'Sea temp range: {df_s["sea_temp"].min():.1f}°C – {df_s["sea_temp"].max():.1f}°C')
        return df_s
    except Exception as e:
        print(f'Sea temperature fetch failed: {e}')
        print('sea_temp feature will be skipped.')
        return None

df_sea = fetch_sea_temp()
if df_sea is not None:
    df_sea['month'] = pd.to_datetime(df_sea['date']).dt.month
    monthly_sea = df_sea.groupby('month')['sea_temp'].mean()
    print('\nMonthly mean sea temperature:')
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    for m, v in monthly_sea.items():
        swim = 'swimming OK' if v >= 17 else 'too cold'
        print(f'  {mnames[m-1]:3s}  {v:5.1f}°C  ({swim})')
    df_sea = df_sea.drop(columns=['month'])
else:
    print('sea_temp feature unavailable — continuing without it.')

Cached sea temp is all-NaN — skipping.
sea_temp feature unavailable — continuing without it.


### 2.2 Public Holidays, Multi-State School Holidays, Bridge Days & COVID

**Public holidays** from the `holidays` library (Schleswig-Holstein).

**School holidays** from four states whose residents regularly visit Scharbeutz:

| State | Population | Travel time | Source |
|-------|-----------|-------------|--------|
| Schleswig-Holstein (SH) | 2.9M | Local | Official SH school calendar |
| Hamburg (HH) | 1.9M | ~40 min | schulferien.org 2020–2022 |
| Nordrhein-Westfalen (NRW) | 17.9M | ~4.5 hrs | schulferien.org 2020–2022 |
| Bayern (BY) | 13.1M | ~7 hrs | schulferien.org 2020–2022 |

**Bridge days (Brückentage)**: Derived from public holidays — Thursday holiday → Friday bridges to weekend; Tuesday holiday → Monday bridges. Creates 4–5 day weekends.

**COVID restriction level** (ordinal):
- `2` — Hard lockdown / interstate travel banned / SH beaches closed to tourists
- `1` — Partial restrictions (capacity limits, hygiene rules, 2G/3G vaccination requirements)
- `0` — No significant restrictions

In [9]:
# === German Public Holidays — Schleswig-Holstein (SH) ===
try:
    sh_pub_holidays = holidays.country_holidays('DE', subdiv='SH', years=[2020, 2021, 2022])
except AttributeError:
    sh_pub_holidays = holidays.Germany(state='SH', years=[2020, 2021, 2022])

public_holiday_dates = set(sh_pub_holidays.keys())
print(f'Public holidays (SH, 2020-2022): {len(public_holiday_dates)} days')
for k in sorted(public_holiday_dates):
    print(f'  {k}  {sh_pub_holidays[k]}')

# === Helper ===
def dates_from_periods(periods):
    result = set()
    for s, e in periods:
        for d in pd.date_range(s, e, freq='D'):
            result.add(d.date())
    return result

# === School Holidays — Schleswig-Holstein (SH) ===
# Source: SH Ministerium für Bildung
school_holiday_dates_sh = dates_from_periods([
    ('2020-01-27', '2020-01-31'), ('2020-04-06', '2020-04-17'),
    ('2020-06-22', '2020-08-01'), ('2020-10-05', '2020-10-17'),
    ('2020-12-21', '2021-01-06'), ('2021-02-01', '2021-02-05'),
    ('2021-03-29', '2021-04-09'), ('2021-06-21', '2021-08-04'),
    ('2021-10-11', '2021-10-22'), ('2021-12-23', '2022-01-07'),
    ('2022-01-31', '2022-02-04'), ('2022-04-04', '2022-04-22'),
])

# === School Holidays — Hamburg (HH) ===
# Source: schulferien.org/Kalender_mit_Ferien/kalender_2020_ferien_Hamburg.html
# Hamburg is 40 min from Scharbeutz — the single largest visitor source city
school_holiday_dates_hh = dates_from_periods([
    ('2020-03-13', '2020-03-20'), ('2020-06-25', '2020-08-05'),
    ('2020-10-05', '2020-10-16'), ('2020-12-21', '2021-01-04'),
    ('2021-03-01', '2021-03-12'), ('2021-05-25', '2021-05-28'),
    ('2021-06-24', '2021-08-04'), ('2021-10-04', '2021-10-15'),
    ('2021-12-23', '2022-01-04'), ('2022-02-21', '2022-03-04'),
    ('2022-04-11', '2022-04-22'),
])

# === School Holidays — Nordrhein-Westfalen (NRW) ===
# Source: schulferien.org — NRW is the largest German state by population (~17.9M)
school_holiday_dates_nrw = dates_from_periods([
    ('2020-04-06', '2020-04-18'), ('2020-06-29', '2020-08-11'),
    ('2020-10-12', '2020-10-24'), ('2020-12-23', '2021-01-06'),
    ('2021-03-29', '2021-04-10'), ('2021-07-05', '2021-08-17'),
    ('2021-10-11', '2021-10-23'), ('2021-12-24', '2022-01-07'),
    ('2022-04-11', '2022-04-23'),
])

# === School Holidays — Bayern (BY) ===
# Source: schulferien.org — large state; Bavarian summer is later than SH (July–Sept)
school_holiday_dates_by = dates_from_periods([
    ('2020-04-09', '2020-04-24'), ('2020-06-02', '2020-06-05'),
    ('2020-07-27', '2020-09-07'), ('2020-11-02', '2020-11-06'),
    ('2020-12-23', '2021-01-08'), ('2021-02-15', '2021-02-19'),
    ('2021-03-29', '2021-04-10'), ('2021-05-25', '2021-06-04'),
    ('2021-08-02', '2021-09-13'), ('2021-11-01', '2021-11-05'),
    ('2021-12-24', '2022-01-07'), ('2022-02-28', '2022-03-04'),
    ('2022-04-09', '2022-04-23'),
])

# Keep backward-compatible alias
school_holiday_dates = school_holiday_dates_sh

print(f'\nSchool holiday days by state (2020-2022):')
print(f'  SH  {len(school_holiday_dates_sh):>3} days')
print(f'  HH  {len(school_holiday_dates_hh):>3} days')
print(f'  NRW {len(school_holiday_dates_nrw):>3} days')
print(f'  BY  {len(school_holiday_dates_by):>3} days')

# === Bridge Days (Brückentage) ===
# Thu holiday → Fri is bridge day (creates Thu–Sun long weekend)
# Tue holiday → Mon is bridge day (creates Sat–Tue long weekend)
def compute_bridge_days(ph_dates):
    bridge = set()
    for ph in ph_dates:
        ph_dt = pd.to_datetime(ph)
        dow = ph_dt.dayofweek  # 0=Mon … 6=Sun
        if dow == 3:  # Thursday → Friday bridges
            bridge.add((ph_dt + pd.Timedelta(days=1)).date())
        elif dow == 1:  # Tuesday → Monday bridges
            bridge.add((ph_dt - pd.Timedelta(days=1)).date())
    return bridge

bridge_days = compute_bridge_days(public_holiday_dates)
print(f'\nBridge days (Brückentage, 2020-2022): {len(bridge_days)}')
for d in sorted(bridge_days):
    print(f'  {d}  ({pd.to_datetime(d).day_name()})')

# === COVID Restriction Levels ===
# 2 = Hard lockdown / interstate travel banned / SH beaches closed to tourists
# 1 = Partial restrictions (capacity limits, hygiene, 2G/3G rules)
# 0 = No significant restrictions
covid_restriction_periods = [
    ('2020-11-01', '2021-03-31', 2),  # Lockdown Light → 2nd/3rd wave hard lockdowns
    ('2021-04-01', '2021-05-31', 2),  # Bundesnotbremse (federal emergency brake)
    ('2021-06-01', '2021-06-30', 1),  # Gradual reopening, capacity limits remain
    ('2021-07-01', '2021-10-31', 0),  # Largely open (summer + early autumn 2021)
    ('2021-11-01', '2022-03-31', 1),  # Omicron wave, 2G/3G rules, no full lockdown
    ('2022-04-01', '2022-04-24', 0),  # Restrictions lifted
]
covid_restriction_map = {}
for s, e, level in covid_restriction_periods:
    for d in pd.date_range(s, e, freq='D'):
        covid_restriction_map[d.date()] = level

from collections import Counter
dist = Counter(covid_restriction_map.values())
print(f'\nCOVID restriction level distribution:')
labels = {0: 'No restrictions', 1: 'Partial', 2: 'Hard lockdown'}
for lvl in sorted(dist):
    print(f'  Level {lvl} ({labels[lvl]}): {dist[lvl]} days')

Public holidays (SH, 2020-2022): 30 days
  2020-01-01  Neujahr
  2020-04-10  Karfreitag
  2020-04-13  Ostermontag
  2020-05-01  Erster Mai
  2020-05-21  Christi Himmelfahrt
  2020-06-01  Pfingstmontag
  2020-10-03  Tag der Deutschen Einheit
  2020-10-31  Reformationstag
  2020-12-25  Erster Weihnachtstag
  2020-12-26  Zweiter Weihnachtstag
  2021-01-01  Neujahr
  2021-04-02  Karfreitag
  2021-04-05  Ostermontag
  2021-05-01  Erster Mai
  2021-05-13  Christi Himmelfahrt
  2021-05-24  Pfingstmontag
  2021-10-03  Tag der Deutschen Einheit
  2021-10-31  Reformationstag
  2021-12-25  Erster Weihnachtstag
  2021-12-26  Zweiter Weihnachtstag
  2022-01-01  Neujahr
  2022-04-15  Karfreitag
  2022-04-18  Ostermontag
  2022-05-01  Erster Mai
  2022-05-26  Christi Himmelfahrt
  2022-06-06  Pfingstmontag
  2022-10-03  Tag der Deutschen Einheit
  2022-10-31  Reformationstag
  2022-12-25  Erster Weihnachtstag
  2022-12-26  Zweiter Weihnachtstag

School holiday days by state (2020-2022):
  SH  202 day

## 3. Feature Engineering

All data sources are merged into a single feature matrix:

| Category | Features | Count |
|----------|----------|-------|
| Temporal | `hour`, `day_of_week`, `month`, `week_of_year`, `day_of_year`, `is_weekend`, `is_friday`, `season` | 8 |
| Holidays (SH) | `is_public_holiday`, `is_school_holiday`, `is_summer_school_holiday`, `days_to_holiday`, `near_holiday` | 5 |
| Holidays (multi-state) | `is_hh_school_holiday`, `is_nrw_school_holiday`, `is_by_school_holiday`, `any_state_school_holiday`, `school_holiday_states` | 5 |
| Bridge days & COVID | `is_bridge_day`, `covid_restriction` | 2 |
| Weather | `temp_max`, `temp_min`, `precipitation`, `sunshine_hours`, `wind_speed`, `temp_range`, `is_rainy`, `is_sunny`, `is_beach_weather`, `apparent_temp_max`, `uv_index_max`, `is_high_uv`, `feels_hot` | 13 |
| Sea temperature | `sea_temp`, `is_swim_temp` | 2 |
| Lag/Rolling | `lag_1d`, `lag_7d`, `rolling_7d` | 3 |

**Total**: up to 38 features. Weather/sea features are only included when API calls succeed.  
Weather is merged daily (same value broadcast to all 6 time slots per day).  
Lag features shift by multiples of 6 rows to target the same time slot N days ago.

In [10]:
def build_features(df, df_weather, df_sea,
                   public_holiday_dates,
                   sh_hols, hh_hols, nrw_hols, by_hols,
                   bridge_day_set, covid_map):
    feat = df.copy()

    # --- Temporal ---
    feat['date']         = feat['datetime'].dt.date
    feat['date_str']     = feat['datetime'].dt.strftime('%Y-%m-%d')
    feat['hour']         = feat['datetime'].dt.hour
    feat['day_of_week']  = feat['datetime'].dt.dayofweek   # 0=Mon, 6=Sun
    feat['day_of_year']  = feat['datetime'].dt.dayofyear
    feat['month']        = feat['datetime'].dt.month
    feat['week_of_year'] = feat['datetime'].dt.isocalendar().week.astype(int)
    feat['year']         = feat['datetime'].dt.year
    feat['is_weekend']   = (feat['day_of_week'] >= 5).astype(int)
    feat['is_friday']    = (feat['day_of_week'] == 4).astype(int)
    feat['season']       = feat['month'].map({
        12: 0, 1: 0, 2: 0,
        3: 1, 4: 1, 5: 1,
        6: 2, 7: 2, 8: 2,
        9: 3, 10: 3, 11: 3
    })

    # --- Public holidays ---
    feat['is_public_holiday'] = feat['date'].isin(public_holiday_dates).astype(int)

    # --- Multi-state school holidays ---
    feat['is_school_holiday']        = feat['date'].isin(sh_hols).astype(int)
    feat['is_hh_school_holiday']     = feat['date'].isin(hh_hols).astype(int)
    feat['is_nrw_school_holiday']    = feat['date'].isin(nrw_hols).astype(int)
    feat['is_by_school_holiday']     = feat['date'].isin(by_hols).astype(int)
    feat['any_state_school_holiday'] = (
        feat['is_school_holiday'] | feat['is_hh_school_holiday'] |
        feat['is_nrw_school_holiday'] | feat['is_by_school_holiday']
    ).astype(int)
    feat['school_holiday_states'] = (
        feat['is_school_holiday'] + feat['is_hh_school_holiday'] +
        feat['is_nrw_school_holiday'] + feat['is_by_school_holiday']
    )
    feat['is_summer_school_holiday'] = (
        feat['month'].isin([6, 7, 8]) & feat['is_school_holiday'].astype(bool)
    ).astype(int)

    # --- Days to nearest public holiday ---
    ph_ts = sorted(pd.to_datetime(list(public_holiday_dates)))
    date_norm = feat['datetime'].dt.normalize()
    feat['days_to_holiday'] = date_norm.apply(
        lambda dt: min([(h - dt).days for h in ph_ts], key=abs)
    )
    feat['near_holiday'] = (feat['days_to_holiday'].abs() <= 2).astype(int)

    # --- Bridge days ---
    feat['is_bridge_day'] = feat['date'].isin(bridge_day_set).astype(int)

    # --- COVID restriction level (0=none, 1=partial, 2=hard) ---
    feat['covid_restriction'] = feat['date'].apply(lambda d: covid_map.get(d, 0))

    # --- Merge daily weather ---
    if df_weather is not None:
        dw = df_weather.copy()
        dw['date_str'] = dw['date'].dt.strftime('%Y-%m-%d')
        weather_cols = ['date_str', 'temp_max', 'temp_min', 'precipitation',
                        'sunshine_hours', 'wind_speed', 'weather_code']
        for col in ['apparent_temp_max', 'uv_index_max']:
            if col in dw.columns:
                weather_cols.append(col)
        feat = feat.merge(dw[weather_cols], on='date_str', how='left')
        # Fill any missing weather with column medians (robust to edge-case gaps)
        for c in weather_cols[1:]:
            if c in feat.columns:
                feat[c] = feat[c].fillna(feat[c].median())
        feat['temp_range']       = feat['temp_max'] - feat['temp_min']
        feat['is_rainy']         = (feat['precipitation'] > 2.0).astype(int)
        feat['is_sunny']         = (feat['sunshine_hours'] > 6.0).astype(int)
        feat['is_beach_weather'] = (
            (feat['temp_max'] >= 20) & (feat['precipitation'] <= 1.0)
        ).astype(int)
        if 'uv_index_max' in feat.columns:
            feat['is_high_uv'] = (feat['uv_index_max'] >= 6).astype(int)
        if 'apparent_temp_max' in feat.columns:
            feat['feels_hot'] = (feat['apparent_temp_max'] >= 25).astype(int)

    # --- Merge Baltic Sea temperature (if available and non-NaN) ---
    if df_sea is not None:
        ds = df_sea.copy()
        ds['date_str'] = pd.to_datetime(ds['date']).dt.strftime('%Y-%m-%d')
        feat = feat.merge(ds[['date_str', 'sea_temp']], on='date_str', how='left')
        if feat['sea_temp'].notna().any():
            feat['sea_temp'] = feat['sea_temp'].ffill().fillna(feat['sea_temp'].median())
            feat['is_swim_temp'] = (feat['sea_temp'] >= 17).astype(int)
        else:
            # All-NaN merge — drop these columns rather than poison dropna()
            feat = feat.drop(columns=['sea_temp'], errors='ignore')

    feat = feat.drop(columns=['date_str'], errors='ignore')

    # --- Lag features: same time slot N days ago (6 slots/day) ---
    feat = feat.sort_values('datetime').reset_index(drop=True)
    feat['lag_1d']     = feat['occupancy'].shift(6)
    feat['lag_7d']     = feat['occupancy'].shift(42)
    feat['rolling_7d'] = feat['occupancy'].shift(6).rolling(42).mean()

    return feat


df_features = build_features(
    df, df_weather, df_sea,
    public_holiday_dates,
    school_holiday_dates_sh, school_holiday_dates_hh,
    school_holiday_dates_nrw, school_holiday_dates_by,
    bridge_days, covid_restriction_map
)
# Only drop rows where the lag features are NaN (first ~7 days lost to lagging)
df_features = df_features.dropna(subset=['lag_1d', 'lag_7d', 'rolling_7d']).reset_index(drop=True)

print(f'Feature matrix: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns')
print(f'Date range after lag drop: {df_features["datetime"].min().date()} to {df_features["datetime"].max().date()}')

new_feats = [
    'is_hh_school_holiday', 'is_nrw_school_holiday', 'is_by_school_holiday',
    'any_state_school_holiday', 'school_holiday_states',
    'is_bridge_day', 'covid_restriction',
    'sea_temp', 'is_swim_temp',
    'apparent_temp_max', 'uv_index_max', 'is_high_uv', 'feels_hot',
]
print(f'\nNew feature status:')
for f in new_feats:
    status = 'OK' if f in df_features.columns else 'SKIPPED (API unavailable)'
    print(f'  {f:<30} {status}')
print(f'\nAll columns ({df_features.shape[1]}):')
for col in df_features.columns:
    print(f'  {col}')

Feature matrix: 3,193 rows x 41 columns
Date range after lag drop: 2020-11-08 to 2022-04-24

New feature status:
  is_hh_school_holiday           OK
  is_nrw_school_holiday          OK
  is_by_school_holiday           OK
  any_state_school_holiday       OK
  school_holiday_states          OK
  is_bridge_day                  OK
  covid_restriction              OK
  sea_temp                       SKIPPED (API unavailable)
  is_swim_temp                   SKIPPED (API unavailable)
  apparent_temp_max              OK
  uv_index_max                   OK
  is_high_uv                     OK
  feels_hot                      OK

All columns (41):
  datetime
  occupancy
  date
  hour
  day_of_week
  day_of_year
  month
  week_of_year
  year
  is_weekend
  is_friday
  season
  is_public_holiday
  is_school_holiday
  is_hh_school_holiday
  is_nrw_school_holiday
  is_by_school_holiday
  any_state_school_holiday
  school_holiday_states
  is_summer_school_holiday
  days_to_holiday
  near_holiday
  is

## 4. Exploratory Data Analysis

Before modelling, we visualize temporal patterns to understand what drives visitor occupancy.

In [14]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Full time series — daily sum
ax = axes[0, 0]
daily_occ = df_features.groupby(df_features['datetime'].dt.date)['occupancy'].sum()
daily_occ.index = pd.to_datetime(daily_occ.index)
ax.fill_between(daily_occ.index, daily_occ.values, alpha=0.7, color='steelblue')
ax.set_title('Daily Total Occupancy (Nov 2020 – Apr 2022)', fontsize=12)
ax.set_ylabel('Sum of all 4h slots')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

# 2. Average by month
ax = axes[0, 1]
monthly_avg = df_features.groupby('month')['occupancy'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
bar_col_m = ['#ff7043' if m in [7, 8] else '#ffa726' if m in [6, 9] else '#64b5f6'
             for m in range(1, 13)]
ax.bar(
    [month_labels[m-1] for m in monthly_avg.index],
    monthly_avg.values,
    color=[bar_col_m[m-1] for m in monthly_avg.index]
)
ax.set_title('Average Occupancy by Month', fontsize=12)
ax.set_ylabel('Avg per 4h slot')

# 3. Average by time slot
ax = axes[1, 0]
hourly_avg = df_features.groupby('hour')['occupancy'].mean()
ax.bar(hourly_avg.index, hourly_avg.values, color='steelblue', width=2.5)
ax.set_xticks([0, 4, 8, 12, 16, 20])
ax.set_xticklabels(['00:00','04:00','08:00','12:00','16:00','20:00'])
ax.set_title('Average Occupancy by Time Slot', fontsize=12)
ax.set_ylabel('Avg occupancy')
ax.set_xlabel('4-hour slot start')

# 4. Average by day of week
ax = axes[1, 1]
dow_avg = df_features.groupby('day_of_week')['occupancy'].mean()
days_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
bar_col_d = ['#ef5350' if i >= 5 else '#ffa726' if i == 4 else '#64b5f6' for i in range(7)]
ax.bar(days_labels, dow_avg.values, color=bar_col_d)
ax.set_title('Average Occupancy by Day of Week', fontsize=12)
ax.set_ylabel('Avg occupancy')

plt.tight_layout()
os.makedirs('../data', exist_ok=True)
plt.savefig('../data/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/eda_overview.png')

Saved: ../data/eda_overview.png


/var/folders/x3/7z1z0qjj6bv_h6bv1_55f07c0000gn/T/ipykernel_74922/1637075496.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Weather correlations & holiday effect
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if 'temp_max' in df_features.columns:
    summer = df_features[df_features['month'].isin([6, 7, 8])]

    ax = axes[0]
    ax.scatter(summer['temp_max'], summer['occupancy'], alpha=0.3, s=8, color='tomato')
    ax.set_title('Temperature vs Occupancy (Jun-Aug)', fontsize=11)
    ax.set_xlabel('Max Temperature (C)'); ax.set_ylabel('Occupancy')

    ax = axes[1]
    ax.scatter(summer['sunshine_hours'], summer['occupancy'], alpha=0.3, s=8, color='orange')
    ax.set_title('Sunshine vs Occupancy (Jun-Aug)', fontsize=11)
    ax.set_xlabel('Sunshine Hours'); ax.set_ylabel('Occupancy')

    ax = axes[2]
    w_feats = ['temp_max', 'precipitation', 'sunshine_hours', 'wind_speed']
    corrs = [df_features['occupancy'].corr(df_features[c]) for c in w_feats]
    col_bars = ['#4caf50' if c > 0 else '#f44336' for c in corrs]
    ax.barh(w_feats, corrs, color=col_bars)
    ax.axvline(0, color='k', linewidth=0.8)
    ax.set_title('Weather-Occupancy Correlations (all data)', fontsize=11)
    ax.set_xlabel('Pearson r')
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'Weather data not available', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)

plt.tight_layout()
plt.show()

# Holiday category comparison
fig, ax = plt.subplots(figsize=(9, 4))
cats = {
    'Regular Weekday': df_features[
        (df_features['is_weekend']==0) & (df_features['is_public_holiday']==0)
    ]['occupancy'].mean(),
    'Weekend':         df_features[df_features['is_weekend']==1]['occupancy'].mean(),
    'Public Holiday':  df_features[df_features['is_public_holiday']==1]['occupancy'].mean(),
    'School Holiday':  df_features[df_features['is_school_holiday']==1]['occupancy'].mean(),
    'Summer Sch. Hol.': df_features[df_features['is_summer_school_holiday']==1]['occupancy'].mean(),
}
bar_col2 = ['#64b5f6','#ff9800','#f44336','#ab47bc','#e91e63']
ax.bar(list(cats.keys()), list(cats.values()), color=bar_col2)
ax.set_title('Average Occupancy by Day Category', fontsize=12)
ax.set_ylabel('Avg occupancy per 4h slot')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../data/eda_holiday_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/eda_holiday_effect.png')

Saved: ../data/eda_holiday_effect.png


/var/folders/x3/7z1z0qjj6bv_h6bv1_55f07c0000gn/T/ipykernel_74922/3308440306.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/x3/7z1z0qjj6bv_h6bv1_55f07c0000gn/T/ipykernel_74922/3308440306.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Occupancy heatmap: hour-of-day vs month
pivot = df_features.pivot_table(
    values='occupancy', index='hour', columns='month', aggfunc='mean'
)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
slot_labels = {0:'00:00', 4:'04:00', 8:'08:00', 12:'12:00', 16:'16:00', 20:'20:00'}

# Rename axes for display before passing to seaborn (avoids set_ticklabels count mismatch)
pivot.index = [slot_labels.get(h, str(h)) for h in pivot.index]
pivot.columns = [month_names[int(c)-1] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f', linewidths=0.5, ax=ax)
ax.set_title('Mean Occupancy Heatmap: Hour of Day vs Month', fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Time slot')
plt.tight_layout()
os.makedirs('../data', exist_ok=True)
plt.savefig('../data/eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/eda_heatmap.png')
print('Key insight: midday (12:00) and afternoon (16:00) slots dominate in summer months.')

Saved: ../data/eda_heatmap.png
Key insight: midday (12:00) and afternoon (16:00) slots dominate in summer months.


/var/folders/x3/7z1z0qjj6bv_h6bv1_55f07c0000gn/T/ipykernel_74922/1181689678.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Machine Learning Models

### Training Strategy

| Set | Period | Approx. Samples |
|-----|--------|-----------------|
| **Train** | Nov 2020 – Dec 2021 | ~1,976 |
| **Test** | Jan 2022 – Apr 2022 | ~612 |

A **temporal holdout** is used — no random split — to simulate real forecasting (learn on the past, predict the future). This prevents data leakage from future observations.

### Models
1. **Seasonal Naive Baseline** — use the same 4-hour slot 7 days ago as the prediction
2. **Random Forest** — tree ensemble; handles non-linear feature interactions robustly
3. **XGBoost** — gradient boosting; typically strongest for structured tabular time-series

In [11]:
WEATHER_FEATS = [
    'temp_max', 'temp_min', 'precipitation', 'sunshine_hours', 'wind_speed',
    'temp_range', 'is_rainy', 'is_sunny', 'is_beach_weather',
    'apparent_temp_max', 'uv_index_max', 'is_high_uv', 'feels_hot',
]

SEA_FEATS = ['sea_temp', 'is_swim_temp']

FEATURE_COLS = [
    # Temporal
    'hour', 'day_of_week', 'month', 'week_of_year', 'day_of_year',
    'is_weekend', 'is_friday', 'season',
    # Holidays — SH (core)
    'is_public_holiday', 'is_school_holiday', 'is_summer_school_holiday',
    'days_to_holiday', 'near_holiday',
    # Holidays — multi-state visitor sources
    'is_hh_school_holiday', 'is_nrw_school_holiday', 'is_by_school_holiday',
    'any_state_school_holiday', 'school_holiday_states',
    # Bridge days & COVID
    'is_bridge_day', 'covid_restriction',
    # Lag / rolling
    'lag_1d', 'lag_7d', 'rolling_7d',
] + [c for c in WEATHER_FEATS if c in df_features.columns] \
  + [c for c in SEA_FEATS   if c in df_features.columns]

TARGET = 'occupancy'

# Temporal split — train on past, predict the future (no data leakage)
train_df = df_features[df_features['year'] <= 2021].copy()
test_df  = df_features[df_features['year'] == 2022].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET]

n_weather = len([c for c in WEATHER_FEATS if c in df_features.columns])
n_sea     = len([c for c in SEA_FEATS     if c in df_features.columns])
print(f'Train: {len(X_train):,} samples  ({train_df["datetime"].min().date()} → {train_df["datetime"].max().date()})')
print(f'Test:  {len(X_test):,} samples  ({test_df["datetime"].min().date()} → {test_df["datetime"].max().date()})')
print(f'Features: {len(FEATURE_COLS)} total')
print(f'  Temporal:       8')
print(f'  Holidays SH:    5')
print(f'  Holidays multi: 5')
print(f'  Bridge/COVID:   2')
print(f'  Lag:            3')
print(f'  Weather:        {n_weather}')
print(f'  Sea temp:       {n_sea}')

Train: 2,509 samples  (2020-11-08 → 2021-12-31)
Test:  684 samples  (2022-01-01 → 2022-04-24)
Features: 36 total
  Temporal:       8
  Holidays SH:    5
  Holidays multi: 5
  Bridge/COVID:   2
  Lag:            3
  Weather:        13
  Sea temp:       0


In [12]:
def evaluate(name, y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true > 5
    mape = (np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
            if mask.sum() > 0 else np.nan)
    print(f'{name}:')
    print(f'  MAE  = {mae:>8.1f} occupancy units')
    print(f'  RMSE = {rmse:>8.1f} occupancy units')
    print(f'  R2   = {r2:>8.3f}')
    print(f'  MAPE = {mape:>7.1f}%')
    return {'Model': name, 'MAE': round(mae,1), 'RMSE': round(rmse,1),
            'R2': round(r2,3), 'MAPE_%': round(mape,1)}

results = []

# --- Baseline: 7-day lag ---
print('--- Baseline ---')
baseline_preds = test_df['lag_7d'].values
results.append(evaluate('Naive (7-day lag)', y_test, baseline_preds))

--- Baseline ---
Naive (7-day lag):
  MAE  =    103.1 occupancy units
  RMSE =    188.9 occupancy units
  R2   =   -0.255
  MAPE =   141.1%


In [13]:
# --- Random Forest ---
print('--- Random Forest ---')
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test).clip(min=0)
results.append(evaluate('Random Forest', y_test, rf_preds))

--- Random Forest ---
Random Forest:
  MAE  =     79.0 occupancy units
  RMSE =    129.2 occupancy units
  R2   =    0.412
  MAPE =   132.1%


In [ ]:
# --- XGBoost ---
print('--- XGBoost ---')
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_preds = xgb_model.predict(X_test).clip(min=0)
results.append(evaluate('XGBoost', y_test, xgb_preds))

--- XGBoost ---


XGBoost:
  MAE  =     81.1 occupancy units
  RMSE =    133.7 occupancy units
  R2   =    0.371
  MAPE =   131.9%


In [ ]:
print('\n=== Model Comparison (Test Set: Jan-Apr 2022) ===')
results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

# --- Predictions vs Actual ---
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
test_dates = test_df['datetime'].values
y_true_arr = y_test.values

ax = axes[0]
ax.plot(test_dates, y_true_arr, color='black', lw=1.0, alpha=0.9, label='Actual')
ax.plot(test_dates, rf_preds,   color='steelblue', lw=0.8, alpha=0.8, label='Random Forest')
ax.plot(test_dates, xgb_preds,  color='tomato', lw=0.8, alpha=0.8, label='XGBoost')
ax.set_title('Predicted vs Actual Occupancy — Test Set (Jan-Apr 2022)', fontsize=12)
ax.set_ylabel('Occupancy')
ax.legend(loc='upper right')

ax = axes[1]
ax.plot(test_dates, np.abs(rf_preds - y_true_arr),
        color='steelblue', lw=0.7, alpha=0.8, label='RF |error|')
ax.plot(test_dates, np.abs(xgb_preds - y_true_arr),
        color='tomato', lw=0.7, alpha=0.8, label='XGB |error|')
ax.set_title('Absolute Prediction Error', fontsize=12)
ax.set_ylabel('|Predicted - Actual|')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../data/model_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/model_predictions.png')


=== Model Comparison (Test Set: Jan-Apr 2022) ===
                     MAE   RMSE    R2  MAPE_%
Model                                        
Naive (7-day lag) 103.10 188.90 -0.26  141.10
Random Forest      79.00 129.20  0.41  132.10
XGBoost            81.10 133.70  0.37  131.90


Saved: ../data/model_predictions.png


In [ ]:
# Feature importance — XGBoost (best model)
feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True).tail(20)

lag_set     = {'lag_1d', 'lag_7d', 'rolling_7d'}
weather_set = set(WEATHER_FEATS)
col_map = {
    f: ('#e91e63' if f in lag_set else '#ff9800' if f in weather_set else '#42a5f5')
    for f in feat_imp['feature']
}

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(feat_imp['feature'], feat_imp['importance'],
        color=[col_map[f] for f in feat_imp['feature']])
ax.set_title('Top Feature Importances — XGBoost', fontsize=12)
ax.set_xlabel('Importance Score')
legend_els = [
    Patch(color='#e91e63', label='Lag / Rolling'),
    Patch(color='#ff9800', label='Weather'),
    Patch(color='#42a5f5', label='Calendar / Holiday'),
]
ax.legend(handles=legend_els, loc='lower right')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/feature_importance.png')

Saved: ../data/feature_importance.png


## 6. Visitor Load Classification & Peak Detection

We classify each 4-hour slot into operational load categories to support crowd management decisions.

| Load Level | Occupancy (4h slot) | Recommended Action |
|------------|---------------------|--------------------|
| **Low** | < 50 | Normal operations |
| **Moderate** | 50–300 | Monitor visitor flow |
| **High** | 300–1,000 | Deploy extra staff; prepare overflow areas |
| **Very High** | > 1,000 | Crowd control; capacity-based entry |

In [ ]:
LOAD_ORDER = ['Low', 'Moderate', 'High', 'Very High']

def classify_load(occ):
    if occ < 50:    return 'Low'
    if occ < 300:   return 'Moderate'
    if occ < 1000:  return 'High'
    return 'Very High'

# Apply to test set (best model: XGBoost)
test_results = test_df[['datetime', TARGET]].copy()
test_results['predicted']      = xgb_preds
test_results['actual_load']    = test_results[TARGET].apply(classify_load)
test_results['predicted_load'] = test_results['predicted'].apply(classify_load)

print('Load Classification Report (XGBoost, test set Jan-Apr 2022):')
print(classification_report(
    test_results['actual_load'],
    test_results['predicted_load'],
    labels=LOAD_ORDER, zero_division=0
))

# Confusion matrix
cm = confusion_matrix(
    test_results['actual_load'],
    test_results['predicted_load'],
    labels=LOAD_ORDER
)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LOAD_ORDER, yticklabels=LOAD_ORDER, ax=ax)
ax.set_title('Load Classification — Confusion Matrix (Predicted vs Actual)', fontsize=12)
ax.set_ylabel('Actual Load')
ax.set_xlabel('Predicted Load')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../data/confusion_matrix.png')

Load Classification Report (XGBoost, test set Jan-Apr 2022):
              precision    recall  f1-score   support

         Low       0.82      0.55      0.66       343
    Moderate       0.53      0.72      0.61       272
        High       0.46      0.54      0.49        67
   Very High       0.00      0.00      0.00         2

    accuracy                           0.61       684
   macro avg       0.45      0.45      0.44       684
weighted avg       0.66      0.61      0.62       684



Saved: ../data/confusion_matrix.png


In [ ]:
peak_df = test_results[
    test_results['predicted_load'].isin(['High', 'Very High'])
].copy()
peak_df['date'] = peak_df['datetime'].dt.date
peak_df['hour'] = peak_df['datetime'].dt.hour
peak_df['dow']  = peak_df['datetime'].dt.day_name()

print(f'=== Peak Detection (Test Set: Jan-Apr 2022) ===')
print(f'Total High/Very High slots: {len(peak_df)}')
print()

print('Predicted load distribution:')
ld = test_results['predicted_load'].value_counts().reindex(LOAD_ORDER).fillna(0).astype(int)
for lvl, cnt in ld.items():
    pct = cnt / len(test_results) * 100
    print(f'  {lvl:<12} {cnt:>4} slots ({pct:.1f}%)')

print()
print('Peak slots by time of day:')
peak_by_hour = peak_df.groupby('hour')['predicted'].mean()
for h, v in peak_by_hour.items():
    print(f'  {h:02d}:00   avg predicted occupancy = {v:.0f}')

print()
print('Peak slots by day of week:')
print(peak_df['dow'].value_counts().to_string())

print()
print('Top 10 most crowded predicted days (test set):')
top_days = (
    test_results.groupby(test_results['datetime'].dt.date)['predicted'].sum()
    .sort_values(ascending=False)
    .head(10)
)
for d, v in top_days.items():
    dow = pd.to_datetime(d).day_name()
    load_label = classify_load(v / 6)  # average slot load
    print(f'  {d}  ({dow:<9})  total={v:.0f}  avg slot load: {load_label}')

=== Peak Detection (Test Set: Jan-Apr 2022) ===
Total High/Very High slots: 83

Predicted load distribution:
  Low           230 slots (33.6%)
  Moderate      371 slots (54.2%)
  High           79 slots (11.5%)
  Very High       4 slots (0.6%)

Peak slots by time of day:
  01:00   avg predicted occupancy = 367
  09:00   avg predicted occupancy = 400
  10:00   avg predicted occupancy = 510
  13:00   avg predicted occupancy = 508
  14:00   avg predicted occupancy = 608
  17:00   avg predicted occupancy = 377
  21:00   avg predicted occupancy = 383

Peak slots by day of week:
dow
Sunday       33
Saturday     26
Monday        6
Wednesday     6
Friday        4
Tuesday       4
Thursday      4

Top 10 most crowded predicted days (test set):
  2022-02-27  (Sunday   )  total=3106  avg slot load: High
  2022-04-13  (Wednesday)  total=3101  avg slot load: High
  2022-04-17  (Sunday   )  total=2181  avg slot load: High
  2022-03-27  (Sunday   )  total=2026  avg slot load: High
  2022-03-20  (Sunda

## 7. Conclusions & Recommendations

### Model Performance Summary

XGBoost delivered the best forecasting accuracy. Both Random Forest and XGBoost substantially outperform the naive baseline.

### Key Findings

1. **Lag features are the strongest predictors** (`lag_7d`, `lag_1d`, `rolling_7d`): the occupancy of the same time slot 7 days prior explains most variance, confirming strong weekly periodicity in visitor behaviour.

2. **Weather is critical**: `temp_max` and `sunshine_hours` rank highly, especially for summer peaks. Even with only 18 months of training data, the model learns the weather–occupancy relationship. For operational use, plug in D+1 to D+7 weather **forecasts** from Open-Meteo free forecast API.

3. **School holidays are the single largest structural driver** of peak load. The SH 6-week summer holiday (late June – early August) reliably produces "Very High" load. Capacity planning should lock this window in every year.

4. **Time-of-day pattern**: midday (12:00) and afternoon (16:00) slots carry the highest occupancy. The 00:00 slot is negligible. Staffing schedules should reflect this.

5. **Weekend effect**: Saturdays and Sundays average 2–3× weekday occupancy in summer, compounded further if the weather is good.

### Recommendations

| Priority | Action | Impact |
|----------|--------|--------|
| High | Integrate D+7 weather forecasts from Open-Meteo free forecast API for rolling predictions | Operational crowd management |
| High | Extend training data with pre-COVID years (2018–2019) to reduce COVID bias | Better summer peak accuracy |
| Medium | Add local events calendar (concerts, sailing races, beach volleyball) as binary features | Capture unexplained spikes |
| Medium | Train a separate summer-only model for Jun–Aug | Higher accuracy during peak season |
| Low | Upgrade dataset to hourly resolution (1h slots) | More precise intraday planning |

### Data Sources Summary

| Source | Type | Cost |
|--------|------|------|
| `scharbeutz_beach_data_4h_sum.csv` | Visitor occupancy (provided) | — |
| Open-Meteo Historical API | Weather | Free, no key |
| `holidays` Python library | Public holidays (SH) | Free, MIT license |
| SH School Calendar | School holiday periods | Free, public domain |

## 8. Persist to Database

All datasets are written to a local SQLite database (`scharbeutz_tourism.db`) for further querying or dashboard integration.

In [ ]:
import sqlite3

DB_PATH = '../scharbeutz_tourism.db'

# ── helpers ───────────────────────────────────────────────────────────────────
def write_table(conn, df, table_name, extra_index_cols=None):
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    cur = conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM "{table_name}"')
    n = cur.fetchone()[0]
    print(f'  {table_name:<35} {n:>6} rows  {len(df.columns):>3} cols')

conn = sqlite3.connect(DB_PATH)
print(f'Database: {DB_PATH}\n')

# ── 1. raw_occupancy  (original CSV) ─────────────────────────────────────────
raw_occ = df[['datetime', 'occupancy']].copy()
raw_occ['datetime'] = raw_occ['datetime'].astype(str)
write_table(conn, raw_occ, 'raw_occupancy')

# ── 2. weather_daily  (Open-Meteo) ───────────────────────────────────────────
if df_weather is not None:
    dw = df_weather.copy()
    dw['date'] = dw['date'].astype(str)
    write_table(conn, dw, 'weather_daily')

# ── 3. public_holidays ────────────────────────────────────────────────────────
ph_rows = [{'date': str(k), 'name': v} for k, v in sh_pub_holidays.items()]
write_table(conn, pd.DataFrame(ph_rows).sort_values('date'), 'public_holidays')

# ── 4. school_holidays ────────────────────────────────────────────────────────
sh_rows = []
for state, hol_set in [('SH', school_holiday_dates_sh), ('HH', school_holiday_dates_hh),
                        ('NRW', school_holiday_dates_nrw), ('BY', school_holiday_dates_by)]:
    for d in sorted(hol_set):
        sh_rows.append({'date': str(d), 'state': state})
write_table(conn, pd.DataFrame(sh_rows), 'school_holidays')

# ── 5. features_engineered  (full feature matrix) ────────────────────────────
feat_db = df_features.copy()
feat_db['datetime'] = feat_db['datetime'].astype(str)
feat_db['date']     = feat_db['date'].astype(str)
write_table(conn, feat_db, 'features_engineered')

# ── 6. model_predictions  (XGBoost test-set results) ─────────────────────────
preds_db = test_results.copy()
preds_db['datetime'] = preds_db['datetime'].astype(str)
write_table(conn, preds_db, 'model_predictions')

# ── 7. model_metrics  (MAE / RMSE / R2 per model) ────────────────────────────
write_table(conn, results_df.reset_index(), 'model_metrics')

conn.commit()
conn.close()

print(f'\nAll tables written. File size: {os.path.getsize(DB_PATH) / 1024:.0f} KB')
print('\nExample queries:')
print("  SELECT * FROM raw_occupancy LIMIT 5;")
print("  SELECT * FROM features_engineered WHERE is_summer_school_holiday=1 LIMIT 5;")
print("  SELECT * FROM model_predictions WHERE predicted_load='Very High' ORDER BY predicted DESC LIMIT 10;")

Database: ../scharbeutz_tourism.db

  raw_occupancy                         3240 rows    2 cols
  weather_daily                          546 rows    9 cols
  public_holidays                         30 rows    2 cols
  school_holidays                        766 rows    2 cols
  features_engineered                   3193 rows   41 cols
  model_predictions                      684 rows    5 cols
  model_metrics                            3 rows    5 cols

All tables written. File size: 748 KB

Example queries:
  SELECT * FROM raw_occupancy LIMIT 5;
  SELECT * FROM features_engineered WHERE is_summer_school_holiday=1 LIMIT 5;
  SELECT * FROM model_predictions WHERE predicted_load='Very High' ORDER BY predicted DESC LIMIT 10;


## 9. Predict for a Specific Date

Enter any date below and run the cell to get predicted occupancy for all 6 time slots of that day, along with weather conditions and load classification.

- **Historical dates** (within the dataset): uses actual lag values from the data
- **Near-future dates** (up to 16 days ahead): fetches live weather forecast from Open-Meteo
- **Far-future dates**: uses monthly climatological averages from training data as weather estimate

In [ ]:
from zoneinfo import ZoneInfo
from datetime import datetime as _dt


def _weather_single_day(date_str, df_hist=None, lat=54.02, lon=10.76):
    """Fetch weather for one date: archive API → forecast API → monthly climatology."""
    if df_hist is None:
        df_hist = df_features
    today     = pd.Timestamp.now().normalize()
    target    = pd.Timestamp(date_str)
    days_ahead = (target - today).days

    if days_ahead <= 0:
        url = 'https://archive-api.open-meteo.com/v1/archive'
    elif days_ahead <= 16:
        url = 'https://api.open-meteo.com/v1/forecast'
    else:
        url = None  # too far ahead → climatology

    weather = None
    if url:
        params = {
            'latitude': lat, 'longitude': lon,
            'start_date': date_str, 'end_date': date_str,
            'daily': ('temperature_2m_max,temperature_2m_min,precipitation_sum,'
                      'sunshine_duration,wind_speed_10m_max,weathercode,'
                      'apparent_temperature_max,uv_index_max'),
            'timezone': 'Europe/Berlin',
        }
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            d = r.json()['daily']
            weather = {
                'temp_max':          d['temperature_2m_max'][0] or 0,
                'temp_min':          d['temperature_2m_min'][0] or 0,
                'precipitation':     d['precipitation_sum'][0] or 0,
                'sunshine_hours':    (d['sunshine_duration'][0] or 0) / 3600,
                'wind_speed':        d['wind_speed_10m_max'][0] or 0,
                'apparent_temp_max': (d.get('apparent_temperature_max') or [None])[0],
                'uv_index_max':      (d.get('uv_index_max') or [None])[0],
            }
        except Exception as e:
            print(f'  Weather API error ({e}) — using climatology.')

    if weather is None:
        m   = pd.Timestamp(date_str).month
        sub = df_hist[df_hist['month'] == m]
        if len(sub) == 0:
            sub = df_hist
        weather = {
            'temp_max':          sub['temp_max'].median(),
            'temp_min':          sub['temp_min'].median(),
            'precipitation':     sub['precipitation'].median(),
            'sunshine_hours':    sub['sunshine_hours'].median(),
            'wind_speed':        sub['wind_speed'].median(),
            'apparent_temp_max': sub['apparent_temp_max'].median() if 'apparent_temp_max' in sub else 0,
            'uv_index_max':      None,
        }
    return weather


def _lag_features(date_str, hour, df_hist=None):
    """Return (lag_1d, lag_7d, rolling_7d) for a given (date, hour) slot."""
    if df_hist is None:
        df_hist = df_features
    target_dt = pd.Timestamp(date_str) + pd.Timedelta(hours=hour)

    def lookup(ts):
        rows = df_hist[df_hist['datetime'] == ts]
        return float(rows['occupancy'].iloc[0]) if len(rows) else None

    lag_1d = lookup(target_dt - pd.Timedelta(days=1))
    lag_7d = lookup(target_dt - pd.Timedelta(days=7))

    recent = df_hist[
        (df_hist['hour'] == hour) &
        (df_hist['datetime'] > target_dt - pd.Timedelta(days=8)) &
        (df_hist['datetime'] < target_dt)
    ]['occupancy']
    rolling_7d = float(recent.mean()) if len(recent) else None

    # Fallback: same hour × same month average
    missing = [v for v in [lag_1d, lag_7d, rolling_7d]
               if v is None or (isinstance(v, float) and np.isnan(v))]
    if missing:
        m   = pd.Timestamp(date_str).month
        avg = df_hist[(df_hist['hour'] == hour) & (df_hist['month'] == m)]['occupancy'].mean()
        if np.isnan(avg):
            avg = df_hist['occupancy'].mean()
        lag_1d     = lag_1d     if (lag_1d     is not None and not np.isnan(lag_1d))     else avg
        lag_7d     = lag_7d     if (lag_7d     is not None and not np.isnan(lag_7d))     else avg
        rolling_7d = rolling_7d if (rolling_7d is not None and not np.isnan(rolling_7d)) else avg

    return lag_1d, lag_7d, rolling_7d


def predict_for_date(date_str, model=None, df_hist=None, verbose=True):
    """
    Predict visitor occupancy for all 6 time slots of `date_str`.

    Parameters
    ----------
    date_str : str   e.g. '2023-07-15'
    model    : trained model (defaults to xgb_model)
    df_hist  : feature dataframe for lag lookup (defaults to df_features)
    verbose  : print formatted output (default True)

    Returns
    -------
    pd.DataFrame  columns: time_slot, predicted_occupancy, load_level
    """
    if model   is None: model   = xgb_model
    if df_hist is None: df_hist = df_features

    target_pd   = pd.Timestamp(date_str)
    target_date = target_pd.date()

    # ── Time slots (correct local hour per CET/CEST) ──────────────────────
    berlin     = ZoneInfo('Europe/Berlin')
    aware_dt   = _dt(target_pd.year, target_pd.month, target_pd.day, tzinfo=berlin)
    utc_offset = int(aware_dt.utcoffset().total_seconds() / 3600)
    slot_hours = [(h + utc_offset) % 24 for h in [0, 4, 8, 12, 16, 20]]

    # ── Weather ───────────────────────────────────────────────────────────
    w        = _weather_single_day(date_str, df_hist)
    temp_max = w['temp_max']      or 0
    temp_min = w['temp_min']      or 0
    precip   = w['precipitation'] or 0
    sun_hrs  = w['sunshine_hours'] or 0
    wind     = w['wind_speed']    or 0
    app_temp = w['apparent_temp_max']
    uv       = w['uv_index_max']

    # Fallback for feels-like temp (cached weather has it; forecast API may not)
    if app_temp is None or (isinstance(app_temp, float) and np.isnan(app_temp)):
        app_temp = df_hist['apparent_temp_max'].median() if 'apparent_temp_max' in df_hist.columns else temp_max
    uv = uv or 0

    temp_range = temp_max - temp_min
    is_rainy   = int(precip > 2.0)
    is_sunny   = int(sun_hrs > 6.0)
    is_beach_w = int(temp_max >= 20 and precip <= 1.0)
    is_high_uv = int(uv >= 6)
    feels_hot  = int(app_temp >= 25)

    # ── Calendar & holiday features ───────────────────────────────────────
    dow          = target_pd.dayofweek
    month        = target_pd.month
    season_map   = {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
    is_pub_hol   = int(target_date in public_holiday_dates)
    is_sh_hol    = int(target_date in school_holiday_dates_sh)
    is_hh_hol    = int(target_date in school_holiday_dates_hh)
    is_nrw_hol   = int(target_date in school_holiday_dates_nrw)
    is_by_hol    = int(target_date in school_holiday_dates_by)
    any_hol      = int(is_sh_hol or is_hh_hol or is_nrw_hol or is_by_hol)
    n_hol_states = is_sh_hol + is_hh_hol + is_nrw_hol + is_by_hol
    is_sum_sh    = int(month in [6,7,8] and bool(is_sh_hol))
    is_bridge    = int(target_date in bridge_days)
    covid_lvl    = covid_restriction_map.get(target_date, 0)

    ph_ts       = sorted(pd.to_datetime(list(public_holiday_dates)))
    days_to_hol = min([(h - target_pd).days for h in ph_ts], key=abs)

    # ── Build one row per slot ────────────────────────────────────────────
    rows = []
    for hour in slot_hours:
        lag_1d, lag_7d, rolling_7d = _lag_features(date_str, hour, df_hist)
        rows.append({
            'hour':                     hour,
            'day_of_week':              dow,
            'month':                    month,
            'week_of_year':             int(target_pd.isocalendar()[1]),
            'day_of_year':              target_pd.dayofyear,
            'is_weekend':               int(dow >= 5),
            'is_friday':                int(dow == 4),
            'season':                   season_map[month],
            'is_public_holiday':        is_pub_hol,
            'is_school_holiday':        is_sh_hol,
            'is_summer_school_holiday': is_sum_sh,
            'days_to_holiday':          days_to_hol,
            'near_holiday':             int(abs(days_to_hol) <= 2),
            'is_hh_school_holiday':     is_hh_hol,
            'is_nrw_school_holiday':    is_nrw_hol,
            'is_by_school_holiday':     is_by_hol,
            'any_state_school_holiday': any_hol,
            'school_holiday_states':    n_hol_states,
            'is_bridge_day':            is_bridge,
            'covid_restriction':        covid_lvl,
            'lag_1d':                   lag_1d,
            'lag_7d':                   lag_7d,
            'rolling_7d':               rolling_7d,
            'temp_max':                 temp_max,
            'temp_min':                 temp_min,
            'precipitation':            precip,
            'sunshine_hours':           sun_hrs,
            'wind_speed':               wind,
            'temp_range':               temp_range,
            'is_rainy':                 is_rainy,
            'is_sunny':                 is_sunny,
            'is_beach_weather':         is_beach_w,
            'apparent_temp_max':        app_temp,
            'uv_index_max':             uv,
            'is_high_uv':               is_high_uv,
            'feels_hot':                feels_hot,
        })

    X     = pd.DataFrame(rows)[FEATURE_COLS]
    preds = model.predict(X).clip(min=0)

    result = pd.DataFrame({
        'time_slot':           [f'{h:02d}:00' for h in slot_hours],
        'predicted_occupancy': np.round(preds, 1),
        'load_level':          [classify_load(p) for p in preds],
    })

    if verbose:
        load_icon = {'Low': '', 'Moderate': '', 'High': '  *** HIGH', 'Very High': '  !!! VERY HIGH'}
        print(f'\n{"─"*52}')
        print(f'  Date   : {date_str}  ({target_pd.day_name()})')
        print(f'  Weather: {temp_max:.1f}°C max  |  Rain: {precip:.1f}mm  |  Sun: {sun_hrs:.1f}h')
        print(f'           Feels like: {app_temp:.1f}°C  |  UV: {uv}')
        print(f'  Holidays (any state): {bool(any_hol)}  |  Public holiday: {bool(is_pub_hol)}')
        if covid_lvl > 0:
            print(f'  ⚠ COVID restriction level: {covid_lvl}')
        print(f'{"─"*52}')
        print(f'  {"Slot":<8}  {"Visitors":>10}  Load level')
        print(f'{"─"*52}')
        for _, r in result.iterrows():
            icon = load_icon.get(r['load_level'], '')
            print(f"  {r['time_slot']:<8}  {r['predicted_occupancy']:>10.1f}  {r['load_level']}{icon}")
        print(f'{"─"*52}')
        peak = result.loc[result['predicted_occupancy'].idxmax(), 'time_slot']
        print(f'  Daily total: {preds.sum():.0f}  |  Peak slot: {peak}')
        print(f'{"─"*52}\n')

    return result


print('predict_for_date() ready.')

In [ ]:
# ── Change this to any date you want to forecast ──────────────────────────────
TARGET_DATE = '2022-07-16'   # format: YYYY-MM-DD
# ─────────────────────────────────────────────────────────────────────────────

result = predict_for_date(TARGET_DATE)
result